# Clase 144 — Flash Attention + RoPE + GQA

Los 3 ingredientes que hacen que un LLM moderno (Llama-3, Mistral) corra en GPU consumer.
Todo implementado en numpy puro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

n, d = 32, 64
Q = np.random.randn(n, d) / np.sqrt(d)
K = np.random.randn(n, d) / np.sqrt(d)
V = np.random.randn(n, d)
print('Q,K,V shapes:', Q.shape, K.shape, V.shape)

## 1. Atención naive (Q·K^T → softmax → ·V)

In [ ]:
def attention_naive(Q, K, V):
    scores = Q @ K.T / np.sqrt(Q.shape[-1])   # (n, n)
    # softmax row-wise estable
    scores -= scores.max(axis=-1, keepdims=True)
    p = np.exp(scores)
    p /= p.sum(axis=-1, keepdims=True)
    return p @ V, p

out_naive, attn = attention_naive(Q, K, V)
print(f'out shape: {out_naive.shape}, attn entropy mean: {(-(attn*np.log(attn+1e-9)).sum(-1)).mean():.3f}')
print(f'memoria de la matriz attn: {attn.nbytes} bytes')

## 2. Flash Attention (tiling block-by-block)

La idea clave: nunca materializamos la matriz `(n, n)` completa. Procesamos K, V por bloques y mantenemos un running softmax.

In [ ]:
def attention_flash(Q, K, V, block=8):
    n, d = Q.shape
    O = np.zeros_like(V); L = np.zeros((n, 1)); M = np.full((n, 1), -np.inf)
    scale = 1.0 / np.sqrt(d)
    for j in range(0, n, block):
        Kj = K[j:j+block]; Vj = V[j:j+block]
        s = (Q @ Kj.T) * scale                    # (n, block) — bloque pequeño
        mj = s.max(axis=-1, keepdims=True)
        m_new = np.maximum(M, mj)
        # rescale acumulador con shift de softmax
        alpha = np.exp(M - m_new)
        beta = np.exp(s - m_new)
        L = L * alpha + beta.sum(axis=-1, keepdims=True)
        O = O * alpha + beta @ Vj
        M = m_new
    return O / L

out_flash = attention_flash(Q, K, V, block=8)
print(f'flash vs naive max diff: {np.abs(out_flash - out_naive).max():.2e}')
print(f'memoria pico Flash (1 bloque): {Q.shape[0] * 8 * 8} bytes vs naive {attn.nbytes}')

## 3. RoPE (Rotary Position Embeddings)

Rota pares (Q_2i, Q_2i+1) por ángulo `pos * theta_i`. Propiedad: `<RoPE(q,m), RoPE(k,n)> = f(q, k, m-n)` → solo depende de distancia relativa.

In [ ]:
def rope(x, base=10000):
    n, d = x.shape
    assert d % 2 == 0
    half = d // 2
    theta = 1.0 / (base ** (np.arange(half) / half))
    pos = np.arange(n)[:, None]
    angles = pos * theta[None, :]                # (n, d/2)
    cos, sin = np.cos(angles), np.sin(angles)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = np.empty_like(x)
    out[..., 0::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out

Qr = rope(Q); Kr = rope(K)
print('RoPE aplicado; shapes:', Qr.shape, Kr.shape)
out_rope, _ = attention_naive(Qr, Kr, V)
print(f'rope output mean: {out_rope.mean():.4f}')

## 4. Verificar dependencia solo de distancia relativa

In [ ]:
# Crear q y k constantes, ver que <RoPE(q,m), RoPE(k,n)> depende solo de m-n
q = np.ones((1, d)) * 0.1
k = np.ones((1, d)) * 0.1
results = {}
for m, n_pos in [(0, 5), (3, 8), (10, 15), (0, 1), (5, 6)]:
    qr = rope(np.tile(q, (m+1, 1)))[m:m+1]
    kr = rope(np.tile(k, (n_pos+1, 1)))[n_pos:n_pos+1]
    dp = (qr @ kr.T)[0, 0]
    rel = n_pos - m
    results.setdefault(rel, []).append(dp)
for rel, vals in sorted(results.items()):
    print(f'distancia relativa = {rel}: dot products = {[f"{v:.6f}" for v in vals]} (deberían ser iguales)')

## 5. GQA (Grouped Query Attention)

MHA: cada head tiene su propio (K, V). KV cache = `n_layers * 2 * seq * n_heads * d_head`.
GQA: agrupamos `n_heads` queries en `n_kv_heads` grupos que comparten KV. Llama-3 usa `n_heads=32, n_kv_heads=8`.

In [ ]:
n_heads = 8; n_kv_heads = 2; d_head = 16; seq = 32
# MHA full
K_mha = np.random.randn(n_heads, seq, d_head); V_mha = np.random.randn(n_heads, seq, d_head)
mem_mha = K_mha.nbytes + V_mha.nbytes

# GQA: solo n_kv_heads sets, repetidos n_heads/n_kv_heads veces
K_gqa = np.random.randn(n_kv_heads, seq, d_head); V_gqa = np.random.randn(n_kv_heads, seq, d_head)
mem_gqa = K_gqa.nbytes + V_gqa.nbytes

print(f'MHA KV cache: {mem_mha:,} bytes')
print(f'GQA KV cache: {mem_gqa:,} bytes')
print(f'reducción: {mem_mha / mem_gqa:.1f}x  →  {(1 - mem_gqa/mem_mha)*100:.1f}% menos memoria')

# Group expand: cada grupo sirve n_heads/n_kv_heads heads
group_size = n_heads // n_kv_heads
K_expanded = np.repeat(K_gqa, group_size, axis=0)   # (n_heads, seq, d_head)
print(f'K expandido para atención: {K_expanded.shape}')

## Conclusiones

- **Flash Attention**: tiling + online softmax → O(n) memoria en lugar de O(n²). Hace que seq=128k entre en GPU.
- **RoPE**: position embedding multiplicativo → generaliza a secuencias más largas que las vistas en training (con interp/extrapolación).
- **GQA**: trade-off entre MHA y MQA. Llama-3/Mistral/Gemma usan GQA → KV cache 4-8x más chica → throughput de inference mucho mayor.
- vLLM, TGI y SGLang combinan los 3 + PagedAttention para servir LLMs eficientemente.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Como TensorFlow/PyTorch no están instalados en este entorno, las celdas de deep learning se validan por API (se ejecutan en Colab con GPU); las de **NumPy puro** son autónomas y traen `assert` para verificarse aquí mismo.

### Ejercicio 1 — SDPA: naïve vs tiling (Flash) coinciden (NumPy, verificable)

El *tiling* de Flash Attention nunca materializa la matriz `(n,n)`; con *online softmax* da el **mismo** resultado que la versión naïve. En PyTorch sería `F.scaled_dot_product_attention(Q, K, V)`.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
n, d = 32, 64
Q = rng.standard_normal((n, d)) / np.sqrt(d)
K = rng.standard_normal((n, d)) / np.sqrt(d)
V = rng.standard_normal((n, d))

def attention_naive(Q, K, V):
    s = Q @ K.T / np.sqrt(Q.shape[-1])
    s -= s.max(axis=-1, keepdims=True)
    p = np.exp(s); p /= p.sum(axis=-1, keepdims=True)
    return p @ V

def attention_flash(Q, K, V, block=8):
    n, d = Q.shape
    O = np.zeros_like(V); L = np.zeros((n, 1)); M = np.full((n, 1), -np.inf)
    scale = 1.0 / np.sqrt(d)
    for j in range(0, n, block):
        Kj, Vj = K[j:j+block], V[j:j+block]
        s = (Q @ Kj.T) * scale
        m_new = np.maximum(M, s.max(axis=-1, keepdims=True))
        alpha, beta = np.exp(M - m_new), np.exp(s - m_new)
        L = L * alpha + beta.sum(axis=-1, keepdims=True)
        O = O * alpha + beta @ Vj
        M = m_new
    return O / L

diff = np.abs(attention_flash(Q, K, V) - attention_naive(Q, K, V)).max()
print("max diff flash vs naive:", f"{diff:.2e}")
assert diff < 1e-6      # mismo resultado, O(n) memoria en lugar de O(n^2)
# En PyTorch: torch.nn.functional.scaled_dot_product_attention(Q, K, V)  (usa Flash en GPU)

### Ejercicio 2 — RoPE: dependencia solo de la distancia relativa (NumPy, verificable)

Propiedad `⟨R_θ q, R_φ k⟩ = f(θ − φ)`: mismo `m−n` → mismo producto, sin importar la posición absoluta.

In [ ]:
import numpy as np

def rope(x, base=10000):
    n, d = x.shape; half = d // 2
    theta = 1.0 / (base ** (np.arange(half) / half))
    ang = np.arange(n)[:, None] * theta[None, :]
    cos, sin = np.cos(ang), np.sin(ang)
    out = np.empty_like(x, dtype=float)
    out[:, 0::2] = x[:, 0::2] * cos - x[:, 1::2] * sin
    out[:, 1::2] = x[:, 0::2] * sin + x[:, 1::2] * cos
    return out

d = 64
q = np.ones((1, d)) * 0.1; k = np.ones((1, d)) * 0.1
def dot(m, n):
    qr = rope(np.tile(q, (m+1, 1)))[m:m+1]
    kr = rope(np.tile(k, (n+1, 1)))[n:n+1]
    return float((qr @ kr.T).item())
assert abs(dot(0, 5) - dot(10, 15)) < 1e-9     # ambos con distancia relativa = 5
assert abs(dot(3, 4) - dot(20, 21)) < 1e-9     # ambos con distancia relativa = 1
print("RoPE verificado: <R_m q, R_n k> depende solo de m-n")

### Ejercicio 3 — GQA vs MHA: shapes de K/V (NumPy, verificable)

GQA agrupa `n_heads` queries en `n_kv_heads` grupos que **comparten** K/V. Al atender, se expanden con `np.repeat`. Llama-3: `n_heads=32, n_kv_heads=8`.

In [ ]:
import numpy as np
n_heads, n_kv_heads, d_head, seq = 32, 8, 128, 16
K_gqa = np.random.randn(n_kv_heads, seq, d_head)      # solo 8 sets de K/V
group = n_heads // n_kv_heads                          # 4 queries por grupo
K_exp = np.repeat(K_gqa, group, axis=0)                # expandido para atención
print("K GQA:", K_gqa.shape, "-> expandido:", K_exp.shape)
assert K_exp.shape == (n_heads, seq, d_head)
assert n_heads % n_kv_heads == 0

### Ejercicio 4 — KV cache: MHA vs GQA (NumPy, verificable)

El KV cache domina la memoria en inference larga. GQA lo reduce en el factor `n_heads/n_kv_heads` (4-8×), subiendo el throughput.

In [ ]:
import numpy as np
n_heads, n_kv_heads, d_head, seq = 32, 8, 128, 8192
bytes_por_elem = 2                                     # fp16
def kv_cache_bytes(kv_heads):
    return 2 * kv_heads * seq * d_head * bytes_por_elem   # K y V

mem_mha = kv_cache_bytes(n_heads)
mem_gqa = kv_cache_bytes(n_kv_heads)
print(f"MHA: {mem_mha/1e6:.1f} MB | GQA: {mem_gqa/1e6:.1f} MB | "
      f"reducción {mem_mha/mem_gqa:.0f}x")
assert mem_mha / mem_gqa == n_heads / n_kv_heads

### Ejercicio 5 — FlashAttention v3 en H100 (conceptual)

Sin GPU H100 no se ejecuta; la mejora clave de v3 es aprovechar los Tensor Cores FP8 y el async de Hopper para ~1.5-2× sobre v2.

In [ ]:
# Requiere H100 (arquitectura Hopper) + flash-attn >= 3.
# import torch
# from flash_attn import flash_attn_func
# q = torch.randn(1, 8, 4096, 64, device="cuda", dtype=torch.float16)
# out = flash_attn_func(q, k, v, causal=True)      # v3: FP8 + warp-specialization async
# Benchmark v3 vs v2: ~1.5-2x en H100 por Tensor Cores FP8 y solapamiento de cómputo/IO.
print("FlashAttention v3: FP8 + async de Hopper -> ~1.5-2x sobre v2 (requiere H100)")